# **Unbursted**
## A threshold model to cut off burst events in FCS measurements and estimate free molecule populations
---
### This project is organized in 3 parts:

1. **Simulation engine**: build from scratch a simulation generating timestamps comparable to real FCS data
2. **Free molecules validation**: simulate realistic free-diffusion-only conditions and compare against real experimental data, estimating molecule number via **autocorrelation functions** and **3D-diffusion curve fitting**
3. **Free/bound separation**: simulate realistic mixed populations (e.g. DNA-bound and free EtBr molecules), develop a method to remove burst events, then estimate free molecule number via autocorrelation and **3D-diffusion fitting**

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import os, time, tables
import pycorrelate as pyc
from scipy.optimize import curve_fit
folder_ts = './Pybromo_timestamps'

# I. **free EtBr** comparison

#### In this section, there will be presented a full analysis of both simulated and real data, based **only** on free EtBr solution as the reference model

In [ ]:
detector_1 = np.loadtxt("Real_data/m1/Sol01_m1.asc", usecols=(0,1))
detector_2 = np.loadtxt("Real_data/m2/Sol01_m2.asc", usecols=(0,1))

In [ ]:
ts_01_apd_1 = (detector_1[:,0]/(105*10**6) + 2.44 * 10**(-12) * detector_1[:,1])
# I cut because I want to see only the first second from starting run (a sample)
ts_01_apd_1_cut = ts_01_apd_1[ts_01_apd_1 < 60]
ts_01_apd_2 = (detector_2[:,0]/(105*10**6) + 2.44 * 10**(-12) * detector_2[:,1])
ts_01_apd_2_cut = ts_01_apd_2[ts_01_apd_2 < 60]

with tables.open_file(os.path.join(folder_ts, 'times_dda833_P954_D4.2e-10_6336pM_step5.0us_t_max60.0s_ID0-0.hdf5'), mode='r') as f:
    name = [n for n in f.root.timestamps._v_children
            if not n.endswith('_par')][0]     # first timestamps dataset
    ts = f.root.timestamps[name]
    ts_01_sim = ts.read()
    clk_p = ts.attrs['clk_p'] 
    ts_01_sim = ts_01_sim * clk_p 

#### Compare histograms "by-eye"

In [ ]:
bins = np.arange(0,60,1e-1)
thr = 50 

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(3,1,1)
ax.set_title("Real data experiment: free EtBr")
ax.hist(ts_01_apd_1_cut, bins=bins, color="tab:blue", lw=0.01, label='APD 1')
ax.hlines(thr, 0, 60, 'red')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,2)
ax.hist(ts_01_sim, bins=bins, color="tab:orange", lw=1, label='Simulated data')
ax.hlines(thr, 0, 60, 'red')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,3)
ax.hist(ts_01_apd_1_cut, bins=bins, color="blue", lw=1, label='APD 1')
ax.hist(ts_01_sim, bins=bins, color="red", lw=1, alpha=0.6, label='Simulated data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.draw()

#### Estimate average count rate

In [ ]:
Average_Count_Rate_APD_1 =  ts_01_apd_1_cut.size / ts_01_apd_1_cut.max()
Average_Count_Rate_APD_2 =  ts_01_apd_2_cut.size / ts_01_apd_2_cut.max()
Average_Count_Rate_APD_sim = ts_01_sim.size / ts_01_sim.max() 
print(f"ACR APD 1 [RE]: {Average_Count_Rate_APD_1:.2f}")
print(f"ACR APD 2 [RE]: {Average_Count_Rate_APD_2:.2f}")
print(f"ACR [SM]: {Average_Count_Rate_APD_sim:.2f}")

#### Compute autocorrelation function, fit with 3d-diffusion model

In [ ]:
# PSF of confocal volume
W0 = 0.25e-6
Z0 = 1.0e-6
KAPPA = Z0 / W0  # structure parameter = 4.0

# function: compute G(tau) from two timestamp arrays (ts1=ts2 for ACF)
def compute_G(ts1, ts2, unit, t_max, bins_per_dec=10):
    t1 = np.round(np.sort(ts1) / unit).astype(np.int64)
    t2 = np.round(np.sort(ts2) / unit).astype(np.int64)
    exp_max = int(np.ceil(np.log10(t_max / unit)))
    bins = pyc.make_loglags(0, exp_max, bins_per_dec)
    G = pyc.pcorrelate(t1, t2, bins=bins, normalize=True)
    tau = 0.5 * (bins[1:] + bins[:-1]) * unit
    return tau, G

# function: model to fit - 3D diffusion (gaussian PSF, single population)
def diffusion_3d(tau, tau_diff, A0):
    return 1 + A0 / (1 + tau/tau_diff) / np.sqrt(1 + tau/(KAPPA**2 * tau_diff))

# function: fit computation with above model 
def fit_G(tau, G, guesses, bounds, tau_min=0, label=""):
    mask = tau > tau_min
    popt, pcov = curve_fit(diffusion_3d, tau[mask], G[mask], p0 = guesses, bounds = bounds)
    print(f"[{label}] tau_diff = {popt[0]:.4g} s   A0 = {popt[1]:.4g}")
    return popt, pcov

# Fit and analysis: two ACFs for real data (APD_1 and APD_2) + ACF from simulated data
unit_re, unit_sm = np.diff(ts_01_apd_1).min(), 5e-6   # native time resolutions
unit_re = unit_re * 1.0
T_RE, T_SM = ts_01_apd_1.max(), ts_01_sim.max()

tau1, G1 = compute_G(ts_01_apd_1, ts_01_apd_1, unit_re, T_RE)
tau2, G2 = compute_G(ts_01_apd_2, ts_01_apd_2, unit_re, T_RE)
tau_sm, G_sm = compute_G(ts_01_sim, ts_01_sim, unit_sm, T_SM)

# initial guesses and bounds (reasonable general parameters)
pars=[4e-5, 1.0]
bounds=([1e-7, 1e-3], [1e-2, 1e3])

popt1, pcov1 = fit_G(tau1, G1, pars, bounds, tau_min=5e-6, label="ACF det1")
popt2, pcov2 = fit_G(tau2, G2, pars, bounds, tau_min=5e-6, label="ACF det2")
popt_sm, pcov_sm = fit_G(tau_sm, G_sm, pars, bounds, label="Simulated")

# poptX[0] is the tau_diff from ACFs fit results
# Now the estimated diffusion coefficient will be computed 
D1 = W0**2 / (4 * popt1[0])
D2 = W0**2 / (4 * popt2[0])
D_sm = W0**2 / (4 * popt_sm[0])

print(f"D (det1)     = {D1*1e4:.4g} cm^2/s")   # *1e4 to convert from m^2/s to cm^2/s
print(f"D (det2)     = {D2*1e4:.4g} cm^2/s")
print(f"D (sim) = {D_sm*1e4:.4g} cm^2/s")

#### Plot analysis and compare results

In [ ]:
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(1,1,1)
ax.plot(tau1, G1, color='tab:blue', ms=3, label='Raw ACF det1', lw=0.5)
ax.plot(tau2, G2, color='tab:orange', ms=3, label='Raw ACF det2', lw=0.5 )
ax.plot(tau_sm, G_sm, color='tab:red', ms=3, label='Raw simulated', lw=0.5)
ax.plot(tau1, diffusion_3d(tau1, *popt1), 'b-', label='Fit det1')
ax.plot(tau2, diffusion_3d(tau2, *popt2), color='darkorange', label='Fit det2')
ax.plot(tau_sm, diffusion_3d(tau_sm, *popt_sm), 'r-', label='Fit simulated')
ax.set_xlabel("tau [s]")
ax.set_ylabel("G")
ax.set_ylim(0.8, 1.8)
ax.set_xscale('log')
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()
plt.draw()

#### Calculate n_avg_molecules from fit and compare results

In [ ]:
B_sm = 275.0          # bg_rate from simulation
F_sm = Average_Count_Rate_APD_sim
A0_sm = popt_sm[1]

N_sm = (1 - B_sm/F_sm)**2 / A0_sm
print(f"N_eff (simulated, from fit) = {N_sm:.3g}")

# 954 are the N total molecules generated in simulation
N_eff_expected = 954 * (3.48e-16 / 2.5e-13)
print(f"N_eff expected in N_box: {N_eff_expected:.3g}")

# implicit estimation; because simulation because sim is "adapted" to realistic experiment,
# I assume similar value for real experiment
B_re1 = 275.0  
F_re1 = ts_01_apd_1.size / ts_01_apd_1.max()   # = Average_Count_Rate_APD_1
F_re2 = ts_01_apd_2.size / ts_01_apd_2.max()   # = Average_Count_Rate_APD_2
# same reasoning above, I assume ACR_APD_2 = ~2x ACR_APD_1
B_re2 = 500.0  

N_re_det1 = (1 - B_re1/F_re1)**2 / popt1[1] 
N_re_det2 = (1 - B_re2/F_re2)**2 / popt2[1]

print(f"N_eff (det1) = {N_re_det1:.3g}")
print(f"N_eff (det2) = {N_re_det2:.3g}")

# II. **EtBr-DNA** comparison (0.12 molecules in confocal vol.)

#### The analysis is **incomplete**: actually there is only the histogram comparison and the average count rate. More analysis will be added in future commits.

In [ ]:
detector_1 = np.loadtxt("Real_data/m1/Sol03_m1.asc", usecols=(0,1))
detector_2 = np.loadtxt("Real_data/m2/Sol03_m2.asc", usecols=(0,1))

In [ ]:
ts_03_apd_1 = (detector_1[:,0]/(105*10**6) + 2.44 * 10**(-12) * detector_1[:,1])
# I cut because I want to see only the first second from starting run (a sample)
ts_03_apd_1_cut = ts_03_apd_1[ts_03_apd_1 < 60]
ts_03_apd_2 = (detector_2[:,0]/(105*10**6) + 2.44 * 10**(-12) * detector_2[:,1])
ts_03_apd_2_cut = ts_03_apd_2[ts_03_apd_2 < 60]

with tables.open_file(os.path.join(folder_ts, 'times_8188ec_P954_D4.2e-10_P114_D1e-12_7093pM_step5.0us_t_max60.0s_ID0-0.hdf5'), mode='r') as f:
    name = [n for n in f.root.timestamps._v_children
            if not n.endswith('_par')][0]     # first timestamps dataset
    ts = f.root.timestamps[name]
    ts_03_sim = ts.read()
    clk_p = ts.attrs['clk_p'] 
    ts_03_sim = ts_03_sim * clk_p 

#### Compare histograms "by-eye"

In [ ]:
bins = np.arange(0,60,1e-1)
thr = 350
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(3,1,1)
ax.set_title("Real data experiment: bound EtBr (n avg DNA-EtBr blobs = 0.12)")
ax.hist(ts_03_apd_1_cut, bins=bins, color="tab:blue", lw=0.01, label='APD 1')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.hlines(thr,0, 60, 'red')
ax.legend()
ax = fig.add_subplot(3,1,2)
ax.hist(ts_03_sim, bins=bins, color="tab:orange", lw=1, label='Simulated data')
ax.hlines(thr, 0, 60, 'red')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,3)
ax.hist(ts_03_apd_1_cut, bins=bins, color="blue", lw=1, label='APD 1')
ax.hist(ts_03_sim, bins=bins, color="red", lw=1, alpha=0.6, label='Simulated data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.draw()

#### Estimate average count rate

In [ ]:
Average_Count_Rate_APD_1 =  ts_03_apd_1_cut.size / ts_03_apd_1_cut.max()
Average_Count_Rate_APD_2 =  ts_03_apd_2_cut.size / ts_03_apd_2_cut.max()
Average_Count_Rate_APD_sim = ts_03_sim.size / ts_03_sim.max() 
print(f"ACR APD 1 [RE]: {Average_Count_Rate_APD_1:.2f}")
print(f"ACR APD 2 [RE]: {Average_Count_Rate_APD_2:.2f}")
print(f"ACR [SM]: {Average_Count_Rate_APD_sim:.2f}")

# II. **EtBr-DNA** comparison (0.18 molecules in confocal vol.)

#### The analysis is **incomplete**: actually there is only the histogram comparison and the average count rate. More analysis will be added in future commits.

In [ ]:
detector_1 = np.loadtxt("Real_data/m1/Sol05_m1.asc", usecols=(0,1))
detector_2 = np.loadtxt("Real_data/m2/Sol05_m2.asc", usecols=(0,1))

In [ ]:
ts_05_apd_1 = (detector_1[:,0]/(105*10**6) + 2.44 * 10**(-12) * detector_1[:,1])
# I cut because I want to see only the first second from starting run (a sample)
ts_05_apd_1_cut = ts_05_apd_1[ts_05_apd_1 < 60]
ts_05_apd_2 = (detector_2[:,0]/(105*10**6) + 2.44 * 10**(-12) * detector_2[:,1])
ts_05_apd_2_cut = ts_05_apd_2[ts_05_apd_2 < 60]

with tables.open_file(os.path.join(folder_ts, 'times_efa6aa_P954_D4.2e-10_P171_D1e-12_7472pM_step5.0us_t_max60.0s_ID0-0.hdf5'), mode='r') as f:
    name = [n for n in f.root.timestamps._v_children
            if not n.endswith('_par')][0]     # first timestamps dataset
    ts = f.root.timestamps[name]
    ts_05_sim = ts.read()
    clk_p = ts.attrs['clk_p'] 
    ts_05_sim = ts_05_sim * clk_p 

#### Compare histograms "by-eye"

In [ ]:
bins = np.arange(0,60,1e-1)
thr = 350

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(3,1,1)
ax.set_title("Real data experiment: bound EtBr (n avg DNA-EtBr blobs = 0.18)")
ax.hist(ts_05_apd_1_cut, bins=bins, color="tab:blue", lw=0.01, label='APD 1')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.hlines(thr,0, 60, 'red')
ax.legend()
ax = fig.add_subplot(3,1,2)
ax.hist(ts_05_sim, bins=bins, color="tab:orange", lw=1, label='Simulated data')
ax.hlines(thr, 0, 60, 'red')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
ax = fig.add_subplot(3,1,3)
ax.hist(ts_05_apd_1_cut, bins=bins, color="blue", lw=1, label='APD 1')
ax.hist(ts_05_sim, bins=bins, color="red", lw=1, alpha=0.6, label='Simulated data')
ax.set_xlabel('time [s]')
ax.set_ylabel('Counts')
ax.legend()
plt.draw()

#### Estimate average count rate

In [ ]:
Average_Count_Rate_APD_1 =  ts_05_apd_1_cut.size / ts_05_apd_1_cut.max()
Average_Count_Rate_APD_2 =  ts_05_apd_2_cut.size / ts_05_apd_2_cut.max()
Average_Count_Rate_APD_sim = ts_05_sim.size / ts_05_sim.max() 
print(f"ACR APD 1 [RE]: {Average_Count_Rate_APD_1:.2f}")
print(f"ACR APD 2 [RE]: {Average_Count_Rate_APD_2:.2f}")
print(f"ACR [SM]: {Average_Count_Rate_APD_sim:.2f}")